# B2 — Görev farkında ızgara kodlayıcı (temiz sürüm v2, 9 Eki)
Ressam (ortak küçük GELU ağı, 45K parametre ≈ 45 KB) + pair başına notlar (1/64, 1/32, 1/16 ızgaraları, entropi kodlu).

**Tarif:**
1. **FP32** — `run_pilot` (sıfırdan) / `fp32_devam` (checkpoint'ten, uzun): notlar serbest, bit yok.
2. **Bit aşaması** — `hazirla_rate` + `run_rate`: notlar tamsayı, bit cezası λ. Sert (STE) ya da C3 tarifi (`soft` + `kuma`).
3. **İnce ayar** — `net_ft`: notlar yuvarlanıp dondurulur, sadece ressam eğitilir.

**Kurallar:** her deney hücresi `PILOT` + mimari satırlarıyla başlar · karar son satıra göre (ara satırlar yanıltıcı) · karşılaştırma eşit KB'ta · her koşu yeni etiket.

**Çalıştırma sırası:** 1 → 7 arası bölümleri baştan sona çalıştır, sonra Bölüm 8'den tek bir deney hücresi.

### Durum (9 Eki)
| Koşu | Dist | Notlar | Toplam |
|---|---|---|---|
| p200_uzun (FP32, bit yok) | **0.0898** | — | — |
| Düz yuvarlama (eğitimsiz) | 0.3028 | 433 KB | 0.621 |
| v3c (sert, kısa FP32'den) | 0.2827 | 104 KB | 0.382 |
| **p200u** (sert, uzun FP32'den) | **0.2483** | 94.6 KB | **0.341** |
| p200c3 (C3 tarifi) | ? | ? | ? |

## 1. Kurulum

In [58]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| cihaz:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, f"| bellek: {props.total_memory / 2**30:.1f} GB")

ROOT = Path("/content/drive/MyDrive/deneme")
VIDEO_PATH = Path("/content/drive/MyDrive/0.mkv")
MODEL_DIR = ROOT / "models"
TARGET_CACHE = ROOT / "checkpoints" / "hnerv384_targets_evalrgb_v1.pt"
OUT_DIR = ROOT / "b2"                      # bu notebook'un bütün çıktıları buraya
OUT_DIR.mkdir(exist_ok=True)

for p in (VIDEO_PATH, MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", TARGET_CACHE):
    print("OK " if p.exists() else "YOK", p)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu130 | cihaz: cuda
GPU: NVIDIA A100-SXM4-40GB | bellek: 39.5 GB
OK  /content/drive/MyDrive/0.mkv
OK  /content/drive/MyDrive/deneme/models/posenet.safetensors
OK  /content/drive/MyDrive/deneme/models/segnet.safetensors
OK  /content/drive/MyDrive/deneme/checkpoints/hnerv384_targets_evalrgb_v1.pt


In [59]:
import gc, sys, traceback, types, warnings, torch, torch.nn as nn

KORU = {"distortion_net", "segnet", "posenet", "seg_targets", "pose_targets", "PILOT", "device"}  # silinmeyecekler

def _gpu_tutuyor(o, d=0):
    if torch.is_tensor(o): return o.is_cuda
    if isinstance(o, nn.Module): return any(t.is_cuda for t in [*o.parameters(), *o.buffers()])
    if isinstance(o, torch.optim.Optimizer):
        return any(torch.is_tensor(v) and v.is_cuda for s in o.state.values() for v in s.values()) \
            or any(p.is_cuda for g in o.param_groups for p in g["params"])
    if d < 2 and isinstance(o, dict): return any(_gpu_tutuyor(v, d + 1) for v in list(o.values())[:500])
    if d < 2 and isinstance(o, (list, tuple)): return any(_gpu_tutuyor(v, d + 1) for v in list(o)[:500])
    return False

def _kareleri_bosalt(exc, gorulen=None):
    gorulen = gorulen or set()
    while exc is not None and id(exc) not in gorulen:            # hata zinciri: cause / context
        gorulen.add(id(exc))
        if exc.__traceback__ is not None:
            traceback.clear_frames(exc.__traceback__)            # karelerdeki yerel değişkenleri sil
        _kareleri_bosalt(exc.__cause__, gorulen)
        exc = exc.__context__

def gpu_temizle(*isimler, koru=()):
    """gpu_temizle() → GPU tutan tüm globalleri sil (KORU hariç). İsim verirsen onları da siler."""
    try: torch.cuda.synchronize()                                # yarıda kalan GPU işini bitir
    except Exception: pass
    # 1) donmuş traceback kareleri (Durdur / OOM / hata)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                          # torch'un eski isimleri uyarı basmasın
        for o in gc.get_objects():                               # kim saklarsa saklasın: tüm traceback'ler
            try:
                if isinstance(o, types.TracebackType):
                    traceback.clear_frames(o)                    # karelerin yerellerini boşalt
            except ReferenceError:                               # ölü weakref proxy: dokununca patlıyor, atla
                continue
    ip = get_ipython()
    hatalar = [getattr(sys, "last_exc", None), getattr(sys, "last_value", None)]
    if getattr(ip, "last_execution_result", None) is not None:
        hatalar.append(ip.last_execution_result.error_in_exec)
        ip.last_execution_result.error_in_exec = None
    for e in hatalar:
        if isinstance(e, BaseException): _kareleri_bosalt(e)
    if getattr(sys, "last_traceback", None) is not None:
        traceback.clear_frames(sys.last_traceback)
    sys.last_exc = sys.last_type = sys.last_value = sys.last_traceback = None
    # 2) globaller: verilen isimler + GPU tutan her şey (korunanlar hariç)
    g = ip.user_ns
    sil = set(isimler) | {n for n, v in list(g.items())
                          if not n.startswith("_") and n not in KORU | set(koru) and _gpu_tutuyor(v)}
    for n in sorted(sil):
        g.pop(n, None)
    print("silindi:", ", ".join(sorted(sil)) or "—")
    # 3) Out[] geçmişi ve _ / __ / ___
    ip.run_line_magic("reset", "-f out")
    # 4) topla ve rafı geri ver
    gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()


def gpu_durum(etiket=""):
    a, r = torch.cuda.memory_allocated() / 2**30, torch.cuda.memory_reserved() / 2**30
    t = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"[GPU {etiket}] kullanılan {a:.1f} GB | ayrılmış {r:.1f} GB | toplam {t:.1f} GB")

## 2. Değerlendiriciler ve hedefler (SegNet / PoseNet donmuş)

In [60]:
!uv pip -q install segmentation-models-pytorch timm einops safetensors av

import sys
sys.path.insert(0, str(ROOT))                      # research/ klasörü Drive'da, deneme/ altında
from research.modules import DistortionNet
from research.oracle_pair import seg_preprocess_grad, pose_preprocess_grad

distortion_net = DistortionNet().eval().to(device)
distortion_net.load_state_dicts(MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", device)
distortion_net.requires_grad_(False)               # ağırlıkları asla değişmeyecek

n_seg = sum(p.numel() for p in distortion_net.segnet.parameters())
n_pose = sum(p.numel() for p in distortion_net.posenet.parameters())
print(f"SegNet {n_seg / 1e6:.1f}M | PoseNet {n_pose / 1e6:.1f}M parametre (donmuş)")

SegNet 9.5M | PoseNet 13.9M parametre (donmuş)


In [61]:
targets = torch.load(TARGET_CACHE, map_location="cpu", weights_only=True)
seg_targets = targets["seg_class"].to(device, dtype=torch.uint8)    # [600, 384, 512]: piksel başına sınıf
pose_targets = targets["pose"].to(device, dtype=torch.float32)      # [600, 6]: pair başına kamera hareketi
NUM_PAIRS = seg_targets.shape[0]
print("Seg:", tuple(seg_targets.shape), "| Pose:", tuple(pose_targets.shape), "| pair:", NUM_PAIRS)

counts = torch.bincount(seg_targets.flatten().long(), minlength=5)
for c, n in enumerate(counts.tolist()):
    print(f"  class {c}: %{100 * n / counts.sum().item():6.2f}")
print("Pose std (boyut başına):", [round(v, 4) for v in pose_targets.std(0).tolist()])

Seg: (600, 384, 512) | Pose: (600, 6) | pair: 600
  class 0: % 23.23
  class 1: %  0.59
  class 2: % 49.52
  class 3: %  1.24
  class 4: % 25.43
Pose std (boyut başına): [1.2574, 0.0358, 0.0299, 0.0096, 0.0074, 0.0286]


## 3. Skor (resmi formül, FP32, TF32 kapalı)

In [62]:
import torch.nn.functional as F

def inflate(native):
    """[B, 2, 3, 384, 512] → [B, 2, 3, 874, 1164]. İleri geçişte yuvarlar, geri geçişte gradyanı geçirir."""
    b = native.shape[0]
    up = F.interpolate(native.flatten(0, 1), size=(874, 1164), mode="bicubic", align_corners=False)
    up = up.clamp(0, 255)
    up = up + (up.round() - up).detach()          # STE: ileri = round, geri = özdeşlik
    return up.view(b, 2, 3, 874, 1164)


@torch.no_grad()
def score(render, ids_all, bs=16):
    """render(ids) → [B, 2, 3, 384, 512] (0–255). Resmi ölçümle aynı ayar: FP32, TF32 kapalı."""
    old = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    wrong, pix, pose_sq = 0, 0, 0.0
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        full = inflate(render(ids))
        pred = distortion_net.segnet(seg_preprocess_grad(full)).argmax(1)
        pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
        wrong += int((pred != seg_targets[ids].long()).sum())
        pix += pred.numel()
        pose_sq += float((pose - pose_targets[ids]).square().mean(1).sum())
    torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = old
    seg, pose_t = 100 * wrong / pix, (10 * pose_sq / len(ids_all)) ** 0.5
    return {"dist": seg + pose_t, "seg": seg, "pose": pose_t}


PILOT = torch.arange(200, device=device)         # çalışma ölçeği: ilk 200 pair (tam video 600)

## 4. Model ve loss

In [63]:
import math
import torch.nn as nn
GRID_LEVELS = {64:2, 32:2, 16:2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]

class GridDecoder(nn.Module):
  def __init__(self, n_pairs, levels = GRID_LEVELS, widths = WIDTHS):
    super().__init__()
    self.grids = nn.ParameterDict({str(d): nn.Parameter(torch.zeros(n_pairs, c, 384 // d, 512 // d))
                                  for d, c in levels.items()})

    self.inject = nn.ModuleDict({str(d): nn.Conv2d(c, widths[int(math.log2(64 // d))], 1)
                                     for d, c in levels.items()})
    self.ups = nn.ModuleList(nn.Conv2d(widths[i], 4 * widths[i + 1], 3, padding=1) for i in range(6))
    self.skips = nn.ModuleList(nn.Conv2d(widths[i], widths[i + 1], 1) for i in range(6))
    self.head = nn.Conv2d(widths[-1], 6, 3, padding=1)
  def forward(self, ids, grids = None):
    g = grids if grids is not None else {k: v[ids] for k, v in self.grids.items()}
    x = self.inject["64"](g["64"])
    for i in range(6):
      up = F.pixel_shuffle(self.ups[i](x), 2)
      skip = F.interpolate(self.skips[i](x), scale_factor = 2, mode = "bilinear", align_corners = False)
      x = F.gelu(up + skip)
      d = str(64//2 ** (i+1))
      if d in self.inject:
        x = x + self.inject[d](g[d])
    return (torch.sigmoid(self.head(x)) * 255).view(len(ids), 2, 3, 384, 512)

In [64]:
def l7_loss(logits, truth, tau = 0.3,thr =1.0, mult = 4.0):
  correct = logits.gather(1, truth.unsqueeze(1))
  others = logits.clone()
  others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
  margin = correct - others.amax(1, keepdim = True)
  per_pixel = tau * F.softplus(-margin / tau)

  with torch.no_grad():
    w = 1.0 + mult * (margin < thr).float()
    w = w/w.mean()

  return (per_pixel * w).mean()

def task_loss(native, ids):
  full = inflate(native)
  logits = distortion_net.segnet(seg_preprocess_grad(full))
  pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
  seg_l = 100 * l7_loss(logits, seg_targets[ids].long())
  pose_l = torch.sqrt(10 * F.mse_loss(pose, pose_targets[ids]) + 1e-6)
  return seg_l + pose_l, seg_l.item(), pose_l.item()

## 5. Bit / boyut araçları ve yuvarlama (sert, yumuşak, C3)

In [65]:
BIT2SCORE = 25 / 8 / 37545489                     # 1 bitin skordaki karşılığı
NET_KB = 45                                       # ressam, 8 bit (entropi kodu yalnızca 44.1 KB'a indiriyor)
rate_of_kb = lambda kb: 25 * kb * 1e3 / 37545489  # KB → skor


@torch.no_grad()
def quant_net_(m, bits=8):
    """Ressamın conv ağırlıklarını yerinde, kanal başına simetrik `bits` bite yuvarlar."""
    qmax = 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            s = mod.weight.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            mod.weight.copy_(torch.round(mod.weight / s).clamp(-qmax, qmax) * s)


def entropy_kb(q):
    """Tamsayı dizisinin ampirik entropisi (değerler bağımsız) → KB."""
    _, counts = torch.unique(q, return_counts=True)
    p = counts.double() / q.numel()
    return float(-(counts * p.log2()).sum()) / 8 / 1e3


def laplace_bits(y, b):
    """y değerinin (±0.5 kutusu) Laplace(0, b) altında bit maliyeti.
    DÜZELTME (6 Eki): p_in'deki işaret '+' olmalı; eski sürümde '-' idi → sıfıra yakın değerler 0 bit sayılıyordu."""
    a = y.abs()
    a_in, a_out = a.clamp(max=0.5), a.clamp(min=0.5)
    p_in = 1 - 0.5 * (torch.exp(-(0.5 - a_in) / b) + torch.exp(-(0.5 + a_in) / b))
    p_out = 0.5 * (torch.exp(-(a_out - 0.5) / b) - torch.exp(-(a_out + 0.5) / b))
    return -torch.log2(torch.where(a < 0.5, p_in, p_out).clamp_min(1e-9))


def grids_for(m, ids, train):
    """Eğitimde: ileri geçiş yuvarlanmış değeri görür (STE) + bit tahmini için gürültülü kopya.
       Değerlendirmede: düz yuvarlama."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        x = g[ids]
        fwd[k] = x + (torch.round(x) - x).detach() if train else torch.round(x)
        noisy[k] = x + torch.rand_like(x) - 0.5
    return fwd, noisy


def soft_round(y, alpha):
    """alpha küçük → neredeyse y (düz çizgi); alpha büyük → round(y) (merdiven)."""
    m = torch.floor(y) + 0.5                       # bulunduğu kutunun ortası (ör. 0.3 → 0.5)
    r = y - m                                      # kutu içindeki konum: -0.5 .. +0.5
    return m + torch.tanh(alpha * r) / (2 * math.tanh(alpha / 2))


def grids_for_soft(m, ids, alpha):
    """Eğitimde yumuşak yuvarlama: ressam s(s(y) + gürültü)'yü görür, bit tahmini s(y) + gürültü'den."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        s = soft_round(g[ids], alpha)
        noisy[k] = s + torch.rand_like(s) - 0.5
        fwd[k] = soft_round(noisy[k], alpha)
    return fwd, noisy


def kumaraswamy_noise(shape, a, device):
    """[-0.5, 0.5] aralığında gürültü. a = 1 → düz (bizim eski ±0.5); a > 1 → ortaya yığılmış."""
    b = (2 ** a * (a - 1) + 1) / a                       # tepe tam 0.5'te olsun diye b seçilir
    u = torch.rand(shape, device=device).clamp(1e-6, 1 - 1e-6)
    x = (1 - (1 - u) ** (1 / b)) ** (1 / a)              # ters CDF ile örnekleme → [0, 1]
    return x - 0.5                                       # → [-0.5, 0.5]


def grids_for_c3(m, ids, alpha, a):
    """C3 1. aşama: bit tahmini s(y) + gürültü'den, ressam s(s(y) + gürültü)'yü görür."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        s = soft_round(g[ids], alpha)
        noisy[k] = s + kumaraswamy_noise(s.shape, a, s.device)
        fwd[k] = soft_round(noisy[k], alpha)
    return fwd, noisy

## 6. Bağlamlı entropi modeli (C3 A.1.4 tarzı)
Her not için komşularından (5×5 nedensel maske) ve bir kaba seviyeden μ, σ tahmini → Laplace bit. Sıra 1/64 → 1/32 → 1/16. Şimdilik **sonradan** uygulanıyor (birlikte eğitim notları kötüleştirdi).

In [66]:
class MaskedConv2d(nn.Conv2d):
  def __init__(self, c_in, c_out, k = 5):
    super().__init__(c_in, c_out, k, padding = k//2)
    mask = torch.ones(k, k)
    mask[k // 2, k // 2:] = 0
    mask[k // 2 + 1:,:] = 0
    self.register_buffer("mask", mask.view(1, 1, k, k))
  def forward(self, x):
    return F.conv2d(x, self.weight * self.mask, self.bias, padding = self.padding)

In [67]:
class LevelEntropy(nn.Module):
  def __init__(self, c = 2, h = 16, has_coarse = True):
    super().__init__()
    self.same = MaskedConv2d(c, h, 5)
    self.coarse = nn.Conv2d(c, h, 1) if has_coarse else None
    self.mix = nn.Sequential(
        nn.GELU(), nn.Conv2d(h, h, 1),
        nn.GELU(), nn.Conv2d(h, 2 * c, 1),
    )
    nn.init.zeros_(self.mix[-1].weight)
    nn.init.zeros_(self.mix[-1].bias)
  def forward(self, q, q_coarse):
    x = self.same(q)
    if self.coarse is not None:
      up = F.interpolate(q_coarse, size=q.shape[-2:], mode="nearest")
      x = x + self.coarse(up)
    mu, log_s = self.mix(x).chunk(2, dim = 1)
    return mu, log_s.exp().clamp(0.1)


class ContextEntropy(nn.Module):
  ORDER = ["64", "32", "16"]
  def __init__(self, levels = GRID_LEVELS):
    super().__init__()
    self.nets = nn.ModuleDict({k: LevelEntropy(levels[int(k)], has_coarse=(i > 0))
                            for i, k in enumerate(self.ORDER)})
  def forward(self, q):
    out = {}
    for i, k in enumerate(self.ORDER):
      onceki = q[self.ORDER[i-1]] if i>0 else None
      out[k] = self.nets[k](q[k], onceki)
    return out

In [68]:
ent_kb = lambda e: sum(p.numel() for p in e.parameters()) / 1e3        # entropi ağı ≈ parametre başına 1 bayt

def context_bits(ent, ctx, vals):
    """ctx: bağlam olarak kullanılan yuvarlanmış notlar. vals: biti ölçülen değerler
    (eğitimde gürültülü, ölçümde tamsayı). Döner: {seviye: toplam bit}."""
    tahmin = ent(ctx)
    return {k: laplace_bits(vals[k] - mu, s).sum() for k, (mu, s) in tahmin.items()}


@torch.no_grad()
def context_kb(ent, grids, ids_all, bs=20):
    """Tamsayı notların bağlamlı model altındaki kod uzunluğu, 600 pair'e ölçekli, seviye seviye KB."""
    toplam = {k: 0.0 for k in ContextEntropy.ORDER}
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        q = {k: torch.round(grids[k][ids]) for k in ContextEntropy.ORDER}
        for k, b in context_bits(ent, q, q).items():
            toplam[k] += b.item()
    olcek = NUM_PAIRS / len(ids_all) / 8 / 1e3                       # bit → KB, 600 pair'e ölçekle
    return {k: v * olcek for k, v in toplam.items()}

## 7. Eğitim fonksiyonları

In [69]:
import time
from tqdm.auto import tqdm


def run_pilot(levels, widths, epochs = 300, tag = "", batch = 20, lr_net=1e-3, lr_grid=1e-2, eval_every=100):
  torch.manual_seed(0)                                       # her deney aynı başlangıçla → adil kıyas
  m = GridDecoder(NUM_PAIRS, levels, widths).to(device)
  n_net = sum(p.numel() for n, p in m.named_parameters() if not n.startswith("grids"))
  n_grid = sum(g[0].numel() for g in m.grids.values())
  print(f"[{tag}] ağ {n_net:,} parametre | pair başına {n_grid:,} ızgara değeri")
  net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
  opt = torch.optim.Adam([{"params": net_p, "lr": lr_net},
                          {"params": list(m.grids.parameters()), "lr": lr_grid}])
  total = epochs * math.ceil(len(PILOT) / batch)
  sched = torch.optim.lr_scheduler.LambdaLR(
      opt, lambda s: min(1.0, (s + 1) / 100) * 0.5 * (1 + math.cos(math.pi * min(s / total, 1.0))))
  t0 = time.time()

  for ep in tqdm(range(1, epochs + 1)):
    m.train()
    perm = PILOT[torch.randperm(len(PILOT), device=device)]
    for s in range(0, len(perm), batch):
            ids = perm[s:s + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
    if ep % eval_every == 0:
            r = score(lambda i: m(i), PILOT)
            print(f"[{tag}] E{ep:04d} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) | "
                  f"{(time.time() - t0) / 60:.1f} dk", flush=True)
  torch.save(m.state_dict(), OUT_DIR / f"pilot_{tag}.pt")
  del m, opt
  torch.cuda.empty_cache()
  return r


def fp32_devam(init_path, tag, epochs, batch=20, lr_net=5e-4, lr_grid=5e-3, warm=20, eval_every=250):
    """FP32 checkpoint'ten devam (bit yok): yarım LR'de yeniden ısınma + kosinüs. Her ölçümde kaydeder."""
    torch.manual_seed(0)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(torch.load(init_path, map_location=device))
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    opt = torch.optim.Adam([{"params": net_p, "lr": lr_net},
                            {"params": list(m.grids.parameters()), "lr": lr_grid}])
    adim = math.ceil(len(PILOT) / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    r = score(m, PILOT)
    print(f"[{tag}] başlangıç | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f})")
    t0 = time.time()
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step()
        if (ep + 1) % eval_every == 0:
            r = score(m, PILOT)
            print(f"[{tag}] E{ep + 1:04d} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) "
                  f"| {(time.time() - t0) / 60:.1f} dk", flush=True)
            torch.save(m.state_dict(), OUT_DIR / f"pilot_{tag}.pt")
    return r

In [70]:
def hazirla_rate(fp32_path, tag, step0=None, oran=0.57):
    """FP32 modeli bit aşamasına hazırlar. Adım = oran × notların std'si (ölçekten bağımsız birim).
    Pilotta 0.25 ≈ 0.57·std idi; 200 pair'de std 0.63 → adım ≈ 0.34."""
    sd = torch.load(fp32_path, map_location=device)
    if step0 is None:
        std = torch.cat([sd[f"grids.{k}"][PILOT].flatten() for k in GRID_LEVELS]).std().item()
        step0 = oran * std
    for k in GRID_LEVELS:
        sd[f"grids.{k}"] /= step0
        sd[f"inject.{k}.weight"] *= step0
    log_b = {str(k): torch.zeros(sd[f"grids.{k}"].shape[1], device=device) for k in GRID_LEVELS}
    yol = OUT_DIR / f"pilot_rate_{tag}_init.pt"
    torch.save({"model": sd, "log_b": log_b, "step0": step0}, yol)
    print(f"[{tag}] adım {step0:.3f} → {yol.name}")
    return yol

In [71]:
def run_rate(lam_max, epochs, init_path, tag, ramp_ep=50, eval_every=50, lr_carpan=1.0, grid_carpan=1.0,
             lam_min=3.0, batch=8, warm=50, soft=None, b_min=0.3, ctx=False, kuma=None):
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])
    log_b = nn.ParameterDict({k: nn.Parameter(v.detach().clone().to(device)) for k, v in ck["log_b"].items()})
    ent = ContextEntropy().to(device) if ctx else None
    if ctx and ck.get("ent") is not None:
        ent.load_state_dict(ck["ent"])
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    grid_p = list(m.grids.parameters())
    gruplar = [{"params": net_p, "lr": 3e-4 * lr_carpan},
               {"params": grid_p, "lr": 3e-2 * lr_carpan * grid_carpan},
               {"params": list(log_b.parameters()), "lr": 1e-2 * lr_carpan}]
    if ctx:
        gruplar.append({"params": list(ent.parameters()), "lr": 3e-3 * lr_carpan})
    opt = torch.optim.Adam(gruplar)
    adim = math.ceil(len(PILOT) / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    kirp = net_p + grid_p + (list(ent.parameters()) if ctx else [])
    for ep in tqdm(range(epochs)):
        lam = lam_min + (lam_max - lam_min) * min(1, (ep + 1) / ramp_ep)
        t = ep / max(1, epochs - 1)
        alpha = 1 / (1 / soft[0] + (1 / soft[1] - 1 / soft[0]) * t) if soft else None   # C3: sıcaklık (1/alpha) doğrusal iner
        a_k = kuma[0] + (kuma[1] - kuma[0]) * t if kuma else None                          # Kumaraswamy a: 2 → 1
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            if kuma:
                fwd, noisy = grids_for_c3(m, ids, alpha, a_k)
            elif soft:
                fwd, noisy = grids_for_soft(m, ids, alpha)
            else:
                fwd, noisy = grids_for(m, ids, train=True)
            task, _, _ = task_loss(m(ids, grids=fwd), ids)
            if ctx:
                bits = sum(context_bits(ent, fwd, noisy).values())                 # bağlam: yuvarlanmış, bit: gürültülü
            else:
                bits = sum(laplace_bits(noisy[k], log_b[k].exp().clamp_min(b_min).view(1, -1, 1, 1)).sum()
                           for k in noisy)
            loss = task + lam * bits / len(ids) * NUM_PAIRS * BIT2SCORE
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(kirp, 1.0); opt.step(); sched.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            r = score(lambda ids: m(ids, grids=grids_for(m, ids, train=False)[0]), PILOT)
            q = {k: torch.round(g[PILOT].detach()) for k, g in m.grids.items()}
            kb_iid = sum(entropy_kb(v) for v in q.values()) * NUM_PAIRS / len(PILOT)
            kb = (sum(context_kb(ent, {k: g.detach() for k, g in m.grids.items()}, PILOT).values()) + ent_kb(ent)
                  if ctx else kb_iid)
            sifir = "/".join(f"%{100 * (v == 0).float().mean():.0f}" for v in q.values())
            print(f"[{tag}] E{ep + 1:03d} λ={lam:.0f} | dist {r['dist']:.4f} | ızgara ~{kb:.0f} KB"
                  f"{f' (bağımsız {kb_iid:.0f})' if ctx else ''} | sıfır {sifir} | toplam ≈ {r['dist'] + rate_of_kb(kb + NET_KB):.3f}")
    torch.save({"model": m.state_dict(), "log_b": log_b.state_dict(), "step0": ck.get("step0"),
                "ent": ent.state_dict() if ctx else None}, OUT_DIR / f"pilot_rate_{tag}.pt")
    return r, kb


def net_ft(init_path, tag, epochs=200, batch=8, lr=3e-4, warm=10):
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])
    with torch.no_grad():
        for g in m.grids.values(): g.round_()
    m.grids.requires_grad_(False)
    if ck.get("ent") is not None:
        ent = ContextEntropy().to(device); ent.load_state_dict(ck["ent"])
        kb = sum(context_kb(ent, dict(m.grids.items()), PILOT).values()) + ent_kb(ent)
    else:
        kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids) * NUM_PAIRS / len(PILOT)
    params = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    adim = math.ceil(len(PILOT) / batch)
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
        if (ep + 1) % 50 == 0:
            r = score(m, PILOT)
            print(f"[{tag}] E{ep + 1:03d} | dist {r['dist']:.4f} | ızgara {kb:.0f} KB "
                  f"| toplam {r['dist'] + rate_of_kb(kb + NET_KB):.3f}")
    torch.save({"model": m.state_dict(), "log_b": ck["log_b"], "step0": ck.get("step0"), "ent": ck.get("ent")},
            OUT_DIR / f"pilot_rate_{tag}.pt")
    return m, r, kb

## 8. Deneyler (200 pair)
Her hücre tek başına çalışır (önce Bölüm 1–7). Sonuç satırını deney defterine işle.

### 8.1 p200_uzun — FP32 uzun eğitim (bit yok)
19a'dan (`pilot_p200.pt`, 0.1563) devam, 1500 ep, L4 ~2.5 sa → **E1500 dist 0.0898** (seg 0.0707, pose 0.0191).

### 8.2 Düz yuvarlama ölçümü (eğitimsiz, ~2 dk)
p200_uzun'un notlarını 0.57·std adımıyla yuvarlar → **433 KB, dist 0.3028**. Uzun FP32 modeli yuvarlamaya çok duyarlı.

In [ ]:
PILOT = torch.arange(200, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]
NET_KB = 45

sd = torch.load(OUT_DIR / "pilot_p200_uzun.pt", map_location=device)
std = torch.cat([sd[f"grids.{k}"][PILOT].flatten() for k in GRID_LEVELS]).std().item()
adim = 0.57 * std

m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
m.load_state_dict(sd)
with torch.no_grad():
    for g in m.grids.values():
        g.copy_(torch.round(g / adim) * adim)      # her notu en yakın adım katına yuvarla

kb = sum(entropy_kb(torch.round(m.grids[k][PILOT] / adim)) for k in m.grids) * NUM_PAIRS / len(PILOT)
r = score(m, PILOT)
print(f"DÜZ YUVARLAMA | adım {adim:.3f} | notlar ~{kb:.0f} KB | dist {r['dist']:.4f} "
      f"(seg {r['seg']:.4f}, pose {r['pose']:.4f}) | toplam {r['dist'] + rate_of_kb(kb + NET_KB):.3f}")
del m, sd

### 8.3 p200u — sert referans (v3c tarifi)
λ 30, 600 ep + net_ft 600 + 300, batch 20, L4 ~2.5 sa → **0.2483 / 94.6 KB / toplam 0.341**.

In [ ]:
PILOT = torch.arange(200, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]
NET_KB = 45

B = 20
s = (B / 8) ** 0.5
T = "p200u"
init = hazirla_rate(OUT_DIR / "pilot_p200_uzun.pt", T)
run_rate(30, 600, init, T, ramp_ep=200, eval_every=100,
         batch=B, lr_carpan=s, grid_carpan=0.55 / s, b_min=0.3)
net_ft(OUT_DIR / f"pilot_rate_{T}.pt", f"{T}_ft600", epochs=600, batch=B, lr=3e-4 * s)
_, r, kb = net_ft(OUT_DIR / f"pilot_rate_{T}_ft600.pt", f"{T}_ft900", epochs=300, batch=B, lr=1e-4 * s)
print(f"SONUÇ {T} | dist {r['dist']:.4f} | {kb:.1f} KB | toplam {r['dist'] + rate_of_kb(kb + NET_KB):.3f}")



```
# This is formatted as code
```

### 8.4 p200c3 — C3 tarifi (gece hücresi, L4 ~4.5–5 sa)
B: yumuşak yuvarlama (alpha 3.3→10, T 0.3→0.1) + Kumaraswamy (a 2→1), 1500 ep · C: sert + STE, her şey düşük LR, 300 ep · sonra net_ft 600 + 300.
Her aşama kendi dosyasını kaydeder; kopma olursa son tamamlanan aşamadan devam.

In [72]:
PILOT = torch.arange(600, device=device)          # tam video
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]
NET_KB = 45

B = 40
s = (B / 8) ** 0.5
T = "p600sifir"
init = hazirla_rate(OUT_DIR / "pilot_p600_uzun.pt", T)
ck = torch.load(init, map_location=device)
ck["model"]["grids.64"].zero_()
ck["model"]["grids.32"].zero_()
ck["model"]["grids.16"].zero_()
torch.save(ck, init)



run_rate(30, 1500, init, T, ramp_ep=500, eval_every=250,
         batch=B, lr_carpan=s, grid_carpan=1.65 / s, b_min=0.3,soft=(3.33, 10.0),kuma=(2.0, 1.0))
net_ft(OUT_DIR / f"pilot_rate_{T}.pt", f"{T}_ft600", epochs=600, batch=B, lr=3e-4 * s)
_, r, kb = net_ft(OUT_DIR / f"pilot_rate_{T}_ft600.pt", f"{T}_ft900", epochs=300, batch=B, lr=1e-4 * s)
print(f"SONUÇ {T} | dist {r['dist']:.4f} | {kb:.1f} KB | toplam {r['dist'] + rate_of_kb(kb + NET_KB):.3f}")

[p600sifir] adım 0.401 → pilot_rate_p600sifir_init.pt


  0%|          | 0/1500 [00:00<?, ?it/s]

[p600sifir] E250 λ=16 | dist 0.5496 | ızgara ~146 KB | sıfır %85/%79/%61 | toplam ≈ 0.677
[p600sifir] E500 λ=30 | dist 0.4990 | ızgara ~94 KB | sıfır %92/%87/%70 | toplam ≈ 0.592
[p600sifir] E750 λ=30 | dist 0.4448 | ızgara ~83 KB | sıfır %93/%89/%72 | toplam ≈ 0.530
[p600sifir] E1000 λ=30 | dist 0.4065 | ızgara ~76 KB | sıfır %94/%90/%73 | toplam ≈ 0.487
[p600sifir] E1250 λ=30 | dist 0.3839 | ızgara ~67 KB | sıfır %95/%91/%75 | toplam ≈ 0.458
[p600sifir] E1500 λ=30 | dist 0.3785 | ızgara ~65 KB | sıfır %95/%92/%75 | toplam ≈ 0.452


  0%|          | 0/600 [00:00<?, ?it/s]

[p600sifir_ft600] E050 | dist 0.3817 | ızgara 65 KB | toplam 0.455
[p600sifir_ft600] E100 | dist 0.3913 | ızgara 65 KB | toplam 0.465
[p600sifir_ft600] E150 | dist 0.3727 | ızgara 65 KB | toplam 0.446
[p600sifir_ft600] E200 | dist 0.3537 | ızgara 65 KB | toplam 0.427
[p600sifir_ft600] E250 | dist 0.3552 | ızgara 65 KB | toplam 0.429
[p600sifir_ft600] E300 | dist 0.3535 | ızgara 65 KB | toplam 0.427
[p600sifir_ft600] E350 | dist 0.3455 | ızgara 65 KB | toplam 0.419
[p600sifir_ft600] E400 | dist 0.3455 | ızgara 65 KB | toplam 0.419
[p600sifir_ft600] E450 | dist 0.3386 | ızgara 65 KB | toplam 0.412
[p600sifir_ft600] E500 | dist 0.3372 | ızgara 65 KB | toplam 0.410
[p600sifir_ft600] E550 | dist 0.3367 | ızgara 65 KB | toplam 0.410
[p600sifir_ft600] E600 | dist 0.3364 | ızgara 65 KB | toplam 0.410


  0%|          | 0/300 [00:00<?, ?it/s]

[p600sifir_ft900] E050 | dist 0.3426 | ızgara 65 KB | toplam 0.416
[p600sifir_ft900] E100 | dist 0.3404 | ızgara 65 KB | toplam 0.414
[p600sifir_ft900] E150 | dist 0.3376 | ızgara 65 KB | toplam 0.411
[p600sifir_ft900] E200 | dist 0.3351 | ızgara 65 KB | toplam 0.408
[p600sifir_ft900] E250 | dist 0.3342 | ızgara 65 KB | toplam 0.408
[p600sifir_ft900] E300 | dist 0.3342 | ızgara 65 KB | toplam 0.408
SONUÇ p600sifir | dist 0.3342 | 65.1 KB | toplam 0.408


In [57]:
gpu_temizle()

silindi: ck, counts
Flushing output cache (0 entries)


## 9. Tanılar

In [ ]:
x = kumaraswamy_noise((100000,), 2.0, device)            # sağlama: ortalama ~0, |x|<0.25 oranı ~%73
print(f"a=2: ortalama {x.mean():+.3f} | |x|<0.25 oranı %{100 * (x.abs() < 0.25).float().mean():.0f} (düzde %50)")
del x

In [ ]:
ent = ContextEntropy().to(device)
print("toplam parametre:", sum(p.numel() for p in ent.parameters()))
ck = torch.load(OUT_DIR / "pilot_rate_p60v3_sert_ft900.pt", map_location=device)
q = {k: torch.round(ck["model"][f"grids.{k}"][:8]) for k in ContextEntropy.ORDER}
with torch.no_grad():
    for k, (mu, s) in ent(q).items():
        print(f"1/{k}: not {tuple(q[k].shape)} → μ {tuple(mu.shape)} | μ ort {mu.mean():.3f} | σ ort {s.mean():.3f}")

In [ ]:
gpu_temizle()
gpu_durum()